In [37]:

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "card_transaction.v1.csv"   
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


# client = OpenAI()


In [39]:
DATA_PATH = "../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)


In [40]:

df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

df["Errors?"] = (df["Errors?"] != "None").astype(int)

df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

df = df.fillna(0)

# print(df.dtypes)
# print(df.head())

/tmp/ipykernel_3598242/3282332532.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [41]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [42]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

## Run form here

In [43]:
# STEP 4: Connect to Ollama
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()
    
    if "response" in data:
        return data["response"]
    else:
        print("Error:", data)
        return None

In [44]:

# from dotenv import load_dotenv
# import os
# import google.generativeai as genai

# load_dotenv()

# genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# for m in genai.list_models():
#     print(m.name, "->", m.supported_generation_methods)

# model = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [45]:
# PROMPT = """
# You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds —  directly)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded —  directly)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a Classifier detect fraud.

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users


# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features

# IMPORTANT RULES:

# - Always use "Hour" for time-based features (NOT Time)
# - Merchant City / State → ONLY nunique or count
# - Ensure all outputs are numeric scalars per user

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# Generate 12–15 high-quality fraud detection features.

# Focus on:
# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior
# - error behavior
# - chip usage patterns

# Do NOT compute values.
# Do NOT output anything outside JSON.


# """

In [46]:
PROMPT = """
You are a fraud analytics feature engineering assistant.

The features you generate will be used to train a RandomForestClassifier.

Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

- User → int64 (group key)
- Card → int64 
- Year → int64
- Month → int64
- Day → int64
- Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
- Hour → float64 (0–23, derived from Time — USE THIS)
- Amount → float64 (numeric)
- Use Chip → float64 (1 = Yes, 0 = No)
- Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
- Merchant City → object (categorical — ONLY use nunique/count)
- Merchant State → object (categorical — ONLY use nunique/count)
- Zip → float64 (numeric)
- MCC → int64 (category code — can use nunique/count)
- Errors? → int64 (1 = error, 0 = no error)
- Is Fraud? → int64 (target label — DO NOT USE)

---

YOUR TASK:

Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a RandomForestClassifier detect fraud.

---

STRICT REQUIREMENTS:

- Features must be NUMERIC only (int or float)
- Must be directly usable in sklearn (no preprocessing needed)
- Must use pandas groupby('User')
- MUST be computed ONLY from df
- SAME features for ALL users

---

FEATURE DESIGN FOCUS:

- spending patterns
- transaction frequency
- merchant/category diversity
- geographic spread
- time behavior (USE Hour only)
- error behavior
- chip usage patterns

---

AVOID:

- raw text usage
- using Merchant Name directly
- using Is Fraud
- string operations
- referencing other generated features
- using Time column
- non-numeric outputs

---

CRITICAL IMPLEMENTATION RULES:

- Always use quoted column names  
  → df.groupby('User')['Amount']

- Use double brackets for multiple columns  
  → df.groupby('User')[['Amount', 'Hour']]

- Never use tuple column selection  
  → ❌ ['Amount', 'Hour']

- Never use reset_index() (any form)

- Each feature MUST follow:  
  df.groupby('User')['COLUMN'].AGG()

- Output must contain exactly one numeric value per user

- Do NOT use:
  - .values  
  - reset_index()  
  - df['User'].map(...)  

- If a feature cannot follow the required pattern → DO NOT include it

---

EXAMPLE:

{
  "feature_name": "avg_amount_per_user",
  "pandas_code": "df.groupby('User')['Amount'].mean()",
  "description": "Average transaction amount per user",
  "type": "numeric"
}

---

OUTPUT FORMAT (STRICT JSON):

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "type": "numeric"
  }
]

---

FINAL INSTRUCTIONS:

- Generate 12–15 high-quality fraud detection features
- Do NOT compute values
- Do NOT explain anything
- Do NOT output anything outside JSON
- Ensure VALID JSON (double quotes only)
"""


In [47]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [48]:
response = call_llm(PROMPT)
print("RAW OUTPUT:\n", response)

RAW OUTPUT:
 Here are the 14 features I came up with, following the strict requirements and rules provided:

[
  {
    "feature_name": "avg_amount_per_user",
    "pandas_code": "df.groupby('User')['Amount'].mean()",
    "description": "Average transaction amount per user",
    "type": "numeric"
  },
  {
    "feature_name": "num_transactions_per_user",
    "pandas_code": "df.groupby('User').size().reset_index(name='num_transactions')",
    "description": "Number of transactions per user",
    "type": "numeric"
  },
  {
    "feature_name": "chip_usage_rate_per_user",
    "pandas_code": "df.groupby('User')['Use Chip'].mean()",
    "description": "Chip usage rate per user",
    "type": "numeric"
  },
  {
    "feature_name": "merchant_category_diversity_per_user",
    "pandas_code": "df.groupby('User')['MCC'].nunique().reset_index(name='diversity')",
    "description": "Merchant category diversity per user",
    "type": "numeric"
  },
  {
    "feature_name": "hourly_transaction_frequency_pe

In [49]:
def extract_json(llm_output):
    try:
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []
    


In [50]:
# response = model_llm.generate_content(PROMPT)
# response=fix_code(response)
features = extract_json(response)

user_index = pd.Index(sorted(df['User'].unique()), name='User')
user_df = pd.DataFrame(index=user_index)

def normalize_user_feature(result, df, user_index):
    if not isinstance(result, pd.Series):
        raise ValueError('Not a pandas Series')

    # If the feature is row-aligned, collapse it to one value per user.
    if len(result) == len(df) and result.index.equals(df.index):
        result = pd.Series(result.to_numpy(), index=df['User']).groupby(level=0).first()
    elif result.index.has_duplicates:
        result = result.groupby(level=0).first()

    result = pd.to_numeric(result, errors='coerce')
    return result.reindex(user_index)

for f in features:
    try:
        print('Applying:', f['feature_name'])
        result = eval(f['pandas_code'])
        user_df[f['feature_name']] = normalize_user_feature(result, df, user_index)
    except Exception as e:
        print('Error in', f['feature_name'], ':', e)

user_df['is_fraud_user'] = df.groupby('User')['Is Fraud?'].max().reindex(user_index)
user_df = user_df.fillna(0)


Applying: avg_amount_per_user
Applying: num_transactions_per_user
Error in num_transactions_per_user : Not a pandas Series
Applying: chip_usage_rate_per_user
Applying: merchant_category_diversity_per_user
Error in merchant_category_diversity_per_user : Not a pandas Series
Applying: hourly_transaction_frequency_per_user
Error in hourly_transaction_frequency_per_user : Not a pandas Series
Applying: error_rate_per_user
Applying: zip_distribution_per_user
Error in zip_distribution_per_user : Not a pandas Series
Applying: merchant_city_frequency_per_user
Error in merchant_city_frequency_per_user : Not a pandas Series
Applying: state_distribution_per_user
Error in state_distribution_per_user : Not a pandas Series
Applying: hourly_amount_mean_per_user
Error in hourly_amount_mean_per_user : Not a pandas Series
Applying: chip_usage_percentage_per_user
Error in chip_usage_percentage_per_user : Not a pandas Series
Applying: merchant_category_entropy_per_user


<string>:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
<string>:1: RuntimeWarning: divide by zero encountered in scalar divide


Error in merchant_category_entropy_per_user : Not a pandas Series
Applying: hourly_error_rate_per_user
Error in hourly_error_rate_per_user : Not a pandas Series


In [51]:
print("\nFinal Shape:", user_df.shape)
print(user_df.head())


Final Shape: (2000, 4)
      avg_amount_per_user  chip_usage_rate_per_user  error_rate_per_user  \
User                                                                       
0               81.299989                       0.0                  1.0   
1               81.118050                       0.0                  1.0   
2               35.159687                       0.0                  1.0   
3              117.277603                       0.0                  1.0   
4               97.011698                       0.0                  1.0   

      is_fraud_user  
User                 
0                 1  
1                 1  
2                 1  
3                 1  
4                 1  


In [52]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [53]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [54]:
print(y.isna().sum())

0


In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [56]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [57]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.30      0.26      0.28       131
           1       0.66      0.70      0.68       269

    accuracy                           0.56       400
   macro avg       0.48      0.48      0.48       400
weighted avg       0.54      0.56      0.55       400

ROC-AUC: 0.46265501291183064


In [58]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)

avg_amount_per_user         1.0
chip_usage_rate_per_user    0.0
error_rate_per_user         0.0
dtype: float64


In [59]:
feature_names = [f["feature_name"] for f in features]

In [60]:
top_features = importance.to_dict()

In [61]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}


In [62]:
print(report1)
print(importance)
print(roc1)

              precision    recall  f1-score   support

           0       0.30      0.26      0.28       131
           1       0.66      0.70      0.68       269

    accuracy                           0.56       400
   macro avg       0.48      0.48      0.48       400
weighted avg       0.54      0.56      0.55       400

avg_amount_per_user         1.0
chip_usage_rate_per_user    0.0
error_rate_per_user         0.0
dtype: float64
0.46265501291183064


In [63]:
def get_prompt(llm_input):
  PROMPT = f"""
  You are a fraud analytics feature engineering expert.

  A RandomForestClassifier was trained on USER-LEVEL features.

  ==============================
  DATASET SCHEMA (STRICT)
  ==============================

  - User → int64 (group key)
  - Card → int64 (DO NOT USE)
  - Year → int64
  - Month → int64
  - Day → int64
  - Time → float64 (timestamp in seconds — DO NOT USE directly)
  - Hour → float64 (0–23 — USE THIS for time features)
  - Amount → float64
  - Use Chip → float64 (1 = Yes, 0 = No)
  - Merchant Name → int64 (DO NOT USE directly)
  - Merchant City → object (ONLY use nunique/count)
  - Merchant State → object (ONLY use nunique/count)
  - Zip → float64
  - MCC → int64
  - Errors? → int64 (1 = error, 0 = no error)
  - Is Fraud? → int64 (TARGET — DO NOT USE)

  ==============================
  CURRENT FEATURES
  ==============================
  {llm_input["existing_features"]}

  ==============================
  MODEL PERFORMANCE
  ==============================

  Classification Report:
  {llm_input["classification_report"]}

  ROC-AUC:
  {llm_input["roc_auc"]}

  Top Important Features:
  {llm_input["top_features"]}

  ==============================
  YOUR TASK
  ==============================

  1. Analyze weaknesses in the model (especially fraud recall)
  2. Identify missing behavioral signals
  3. Suggest 8–12 NEW or IMPROVED features

  ==============================
  STRICT RULES (VERY IMPORTANT)
  ==============================

  - Features must be NUMERIC only
  - Must work directly in sklearn (no preprocessing)
  - Must use pandas groupby('User')
  - MUST be directly computable from df
  - MUST NOT depend on other generated features
  - MUST NOT duplicate existing features


  AVOID:
  - raw text usage
  - using Merchant Name directly
  - using Is Fraud?
  - referencing other generated features
  - using Time (use Hour only)

  IMPORTANT:
  - Each pandas_code MUST return a pandas Series indexed by User
  - Do NOT use df['User'].map(...)

  ==============================
  OUTPUT FORMAT (STRICT JSON)
  ==============================

  [
    {{
      "feature_name": "...",
      "pandas_code": "...",
      "reason": "..."
    }}
  ]

  Focus on:
  - behavioral anomalies
  - spending variability
  - temporal patterns (Hour-based)
  - transaction bursts
  - merchant/category diversity
  - geographic spread
  - error behavior
  - chip usage changes

  Do NOT compute values.
  Do NOT output anything outside JSON.
  """
  return PROMPT


In [64]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")
# response = model_llm.generate_content(PROMPT)


In [65]:
a=1
for a in range(1,11):
    PROMPT=get_prompt(llm_input)
    response = call_llm(PROMPT)

    output = response.text.replace("```json", "").replace("```", "").strip()

    new_features = json.loads(output)
    base_features = []

    for f in features:
        code = f["pandas_code"]
        
        # Only keep df-based features
        if "df.groupby" in code and "user_" not in code.split("=")[-1]:
            base_features.append(f)
            
    for f in base_features:
        try:
            user_df[f["feature_name"]] = eval(f["pandas_code"])
        except Exception as e:
            print("Base error:", f["feature_name"], e)
    
            
    a+=1

AttributeError: 'str' object has no attribute 'text'

In [ ]:
base_features = []

for f in features:
    code = f["pandas_code"]
    
    # Only keep df-based features
    if "df.groupby" in code and "user_" not in code.split("=")[-1]:
        base_features.append(f)

In [ ]:
for f in base_features:
    try:
        user_df[f['feature_name']] = normalize_user_feature(eval(f['pandas_code']), df, user_index)
    except Exception as e:
        print('Base error:', f['feature_name'], e)


In [ ]:
X = user_df.drop(columns=["is_fraud_user"])
y = user_df["is_fraud_user"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ================================
# 🔹 11. EVALUATE
# ================================
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== ROC AUC ===")
print(roc_auc_score(y_test, y_prob))

# ================================
# 🔹 12. FEATURE IMPORTANCE
# ================================
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n=== Top Features ===")
print(importance.head(10))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.90      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400


=== ROC AUC ===
0.9027214166122761

=== Top Features ===
user_tx_count                     0.177505
user_city_tx_ratio                0.164542
user_merchant_city_count          0.155953
user_mcc_nunique                  0.116284
user_merchant_city_nunique        0.108388
user_merchant_state_nunique       0.102826
user_std_amount                   0.048543
user_zip_change_std               0.046680
user_avg_amount                   0.040045
user_avg_amount_per_city_ratio    0.039234
dtype: float64


In [ ]:
report2 = classification_report(y_test, y_pred)
roc2 = roc_auc_score(y_test, y_prob)

In [ ]:
# precision    recall  f1-score   support

#           No       0.87      0.72      0.79       131
#          Yes       0.87      0.95      0.91       269

#     accuracy                           0.87       400
#    macro avg       0.87      0.83      0.85       400
# weighted avg       0.87      0.87      0.87       400

# ROC-AUC: 0.9174210391895343

In [ ]:
print(report1)
print(report2)


              precision    recall  f1-score   support

           0       0.88      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.84       400
weighted avg       0.87      0.87      0.87       400

              precision    recall  f1-score   support

           0       0.90      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400



In [ ]:
print(roc1)
print(roc2)

0.9049348732937939
0.9027214166122761
